# **DINO enhanced visualisation**

## **1. Importing**

In [8]:
import torch
import numpy as np
import torchvision
from transformers import AutoImageProcessor, AutoModel
from torch.utils.data import DataLoader
import h5py
import random
import kagglehub
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
import matplotlib.pyplot as plt

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### **1.2 DINO FLAGSHIP MODEL 40 LAYERS WOHOO**

In [36]:
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

# Ensure dtype is defined
if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    #Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)
    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    #Verify Layer Count
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        #Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

Loading flagship model: facebook/dinov3-vit7b16-pretrain-lvd1689m
Targeting Architecture: 40 Layers, 4096 Embedding Dim


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Success! Model loaded with 40 layers.


## **2. Dataset**

### **2.1 CIFAR**

In [38]:
dataset_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
loader_cifar10 = DataLoader(dataset_cifar10, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR10 samples (OOD - CIFAR-10): {len(dataset_cifar10)}")

Amount of CIFAR10 samples (OOD - CIFAR-10): 10000


In [39]:
dataset_cifar100 = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
loader_cifar100 = DataLoader(dataset_cifar100, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of CIFAR100 samples (OOD - CIFAR-100): {len(dataset_cifar100)}")

Amount of CIFAR100 samples (OOD - CIFAR-100): 10000


### **2.2 SVHN**

In [40]:
dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
loader_svhn = DataLoader(dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (Test set): {len(dataset_svhn)}")

Amount of SVHN samples (Test set): 26032


### **2.3 FOOD101**

In [41]:
dataset_food101 = torchvision.datasets.Food101(root='./data', split='test', download=True, transform=transform)
loader_food101 = DataLoader(dataset_food101, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of Food101 samples (test set): {len(dataset_food101)}")

Amount of Food101 samples (test set): 25250


### **2.4 Pokémon**

In [42]:
path = kagglehub.dataset_download("thedagger/pokemon-generation-one")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pokemon-generation-one' dataset.
Path to dataset files: /kaggle/input/pokemon-generation-one


In [43]:
transform_kaggle = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

image_root_path = os.path.join(path, 'dataset')

dataset_pokemon_gen1 = ImageFolder(
    root=image_root_path,
    transform=transform_kaggle
)

loader_pokemon_gen1 = DataLoader(
    dataset_pokemon_gen1,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print(f"\n✅ Success! Loaded {len(dataset_pokemon_gen1)} images.")
print(f"Found classes: {dataset_pokemon_gen1.classes[:5]}...")


✅ Success! Loaded 21314 images.
Found classes: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine']...


## **3. Video Generation**

In [44]:
MODEL_NAME = "facebook/dinov3-vit7b16-pretrain-lvd1689m"
IMAGE_PATH = "your_high_res_image.jpg"  # Replace with your image
OUTPUT_VIDEO = "dinov3_attention_scan.mp4"
PATCH_SIZE = 16  # Standard patch size for ViT models
BIG_PATCH_SIZE = 2  # Combine 2x2 standard patches into one "Big Patch"
STRIDE = 1  # Step size for sliding the Big Patch (in units of standard patches)
FPS = 10  # Frames per second for the video

### **3.1 Implementation**

In [45]:
import requests
import os
from PIL import Image
from io import BytesIO
import cv2

In [46]:
def get_attention_matrix(model, img_tensor):
    """Extracts the self-attention map from the last layer."""
    with torch.no_grad():
        outputs = model(img_tensor, output_attentions=True)

    # Take last layer attention: [batch, heads, tokens, tokens]
    attentions = outputs.attentions[-1]

    # Average across heads and remove batch dim: [tokens, tokens]
    attn_avg = attentions[0].mean(dim=0)

    # Remove CLS token to get patch-to-patch attention
    patch_attn = attn_avg[1:, 1:]
    return patch_attn

In [47]:
def generate_video(key, img, patch_attn):
    """Generates the heatmap video for a specific image."""
    w, h = img.size
    grid_w = w // PATCH_SIZE
    grid_h = h // PATCH_SIZE

    print(f"Processing '{key}': {w}x{h} image ({grid_w}x{grid_h} grid)")

    output_path = os.path.join(OUTPUT_DIR, f"{key}_attention_scan.mp4")
    video = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (w * 2, h))

    # Pre-calculate indices for "Big Patch" logic
    # Shape: [num_patches, num_patches]
    # We move the tensor to CPU to save GPU VRAM for the next steps
    attn_cpu = patch_attn.cpu().float()

    # Reshape for easier indexing: [Grid_H, Grid_W, Total_Patches]
    attn_grid = attn_cpu.view(grid_h, grid_w, -1)

    # Loop with stride
    for y in range(0, grid_h - BIG_PATCH_SIZE + 1, STRIDE):
        for x in range(0, grid_w - BIG_PATCH_SIZE + 1, STRIDE):

            # --- Aggregation Step ---
            # Extract the attention maps for the RxC block of patches
            # We slice the grid to get the query window
            query_block = attn_grid[y : y + BIG_PATCH_SIZE, x : x + BIG_PATCH_SIZE, :]

            # Sum correlations: "What does this whole block look at?"
            # Shape: [Total_Patches]
            heatmap_flat = query_block.sum(dim=(0, 1))

            # --- Visualization ---
            # Reshape back to 2D image grid
            heatmap_2d = heatmap_flat.view(grid_h, grid_w).numpy()

            # Normalize (Min-Max)
            heatmap_norm = (heatmap_2d - heatmap_2d.min()) / (heatmap_2d.max() - heatmap_2d.min() + 1e-8)
            heatmap_uint8 = (heatmap_norm * 255).astype(np.uint8)

            # Resize to original image resolution
            heatmap_overlay = cv2.resize(heatmap_uint8, (w, h), interpolation=cv2.INTER_NEAREST)
            heatmap_color = cv2.applyColorMap(heatmap_overlay, cv2.COLORMAP_JET)

            # Prepare Left Image (Source with Box)
            frame_left = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            cv2.rectangle(
                frame_left,
                (x * PATCH_SIZE, y * PATCH_SIZE),
                ((x + BIG_PATCH_SIZE) * PATCH_SIZE, (y + BIG_PATCH_SIZE) * PATCH_SIZE),
                (0, 255, 0), 3
            )

            # Combine Side-by-Side
            frame_final = np.hstack([frame_left, heatmap_color])
            video.write(frame_final)

    video.release()
    print(f"Saved: {output_path}")

## **4. Running the thing**

In [48]:
IMAGE_URLS = {
    "city_4k": "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?q=80&w=2613&auto=format&fit=crop", # Urban density
    "nature_detail": "https://images.unsplash.com/photo-1501854140884-074bf6b243e7?q=80&w=2525&auto=format&fit=crop", # Organic textures
    "cat_highres": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?q=80&w=2500&auto=format&fit=crop" # Object centric
}

In [49]:
def download_image(url):
    """Downloads an image from a URL and converts it to RGB."""
    print(f"Downloading from {url[:50]}...")
    # Add headers to mimic a browser and avoid 403/406 errors from Unsplash
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return img
    except Exception as e:
        print(f"Failed to download: {e}")
        raise e

In [50]:
for key, url in IMAGE_URLS.items():
    try:
        # A. Download
        raw_img = download_image(url)

        # B. Resize to be divisible by patch size (Keep it High Res!)
        # We limit max dimension to 1024 to avoid OOM on standard GPUs
        # If you have an A100 (80GB), you can increase this to 2048 or 4096
        MAX_DIM = 1024
        scale = min(MAX_DIM / raw_img.width, MAX_DIM / raw_img.height)
        new_w = int((raw_img.width * scale) // PATCH_SIZE * PATCH_SIZE)
        new_h = int((raw_img.height * scale) // PATCH_SIZE * PATCH_SIZE)
        img_resized = raw_img.resize((new_w, new_h))

        # C. Prepare Tensor
        # Manually normalize if processor acts up, or just use processor
        inputs = processor(images=img_resized, return_tensors="pt").to(device)

        # D. Run
        attn_matrix = get_attention_matrix(model, inputs['pixel_values'])
        generate_video(key, img_resized, attn_matrix)

        # E. Cleanup Memory
        del inputs, attn_matrix
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Skipping {key} due to error: {e}")

Processing 'city_4k': 1024x624 image (64x39 grid)
Skipping city_4k due to error: name 'OUTPUT_DIR' is not defined
Failed to download: 404 Client Error: Not Found for url: https://images.unsplash.com/photo-1501854140884-074bf6b243e7?q=80&w=2525&auto=format&fit=crop
Skipping nature_detail due to error: 404 Client Error: Not Found for url: https://images.unsplash.com/photo-1501854140884-074bf6b243e7?q=80&w=2525&auto=format&fit=crop
Processing 'cat_highres': 1024x704 image (64x44 grid)
Skipping cat_highres due to error: name 'OUTPUT_DIR' is not defined
